# Transformer с механизмом копирования

С нуля, тот же BPE8000 и **1 000 000 train / 20 000 val**, seed42.
Архитектура8+8, d_model1024; одна эпоха, effective batch256,
LR2e-4, warmup5% и cosine, валидация каждые1000 шагов и в конце.
T5: GEGLU d_ff2816; copy-контроль: BART FF4096 + отдельная copy-head.

Добавьте приватный dataset с `nlp_hw2_kaggle.zip` через **Add Input**.
Выберите доступный GPU (T4 x2 использует DDP), включите Internet для
установки библиотек. Приватность ноутбука и dataset сохраняйте Private.
Первая проверка обучает отдельную модель2 шага, затем полный запуск
начинает обучение заново с seed42. Готовых весов нет.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

ARCHITECTURE = 'copy'
RUN_NAME = 'copy_008_1m'
MICRO_BATCH = 16       # на GPU; уменьшите до8 при OOM, global batch останется256
GLOBAL_BATCH = 256
LEARNING_RATE = 2e-4
PRECISION = 'auto'     # BF16 при поддержке; иначе FP16 (T4)
USE_COMPILE = False   # опционально; TF32-конфликт исправлен в общем Trainer
GRADIENT_CHECKPOINTING = True
RESUME_FROM = None     # полный путь к прежнему checkpoint-N, при необходимости
EXPECTED_MANIFEST_SHA = '7480c7d1778bffce50db987019f2ccf87c06f8643c1ce59ad811e4fd3c77d0e1'

candidates = list(Path('/kaggle/input').rglob('bundle_manifest.json'))
matches = [p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_MANIFEST_SHA]
if len(matches) != 1:
    raise RuntimeError('Добавьте dataset с подготовленным пакетом этой версии; найдено совпадений: ' + str(len(matches)))
BUNDLE_ROOT = matches[0].parent
OUTPUT = Path('/kaggle/working') / RUN_NAME
print('Bundle:', BUNDLE_ROOT, 'Output:', OUTPUT)


In [ ]:
# Сохраняем рабочий PyTorch/CUDA из образа Kaggle.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
    str(BUNDLE_ROOT / 'requirements-kaggle.txt')], check=True)
sys.path.insert(0, str(BUNDLE_ROOT))
import torch, transformers
print('PyTorch:', torch.__version__, 'Transformers:', transformers.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Выберите GPU в Settings → Accelerator перед запуском.')
NUM_GPUS = min(2, torch.cuda.device_count())
print('GPU:', [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])
from kaggle_experiments import accumulation_steps, load_bundle
print('GPUs:', NUM_GPUS, 'micro batch:', MICRO_BATCH,
    'accumulation:', accumulation_steps(GLOBAL_BATCH, MICRO_BATCH, NUM_GPUS),
    'effective batch:', GLOBAL_BATCH)
data, tokenizer, identity = load_bundle(BUNDLE_ROOT,
    expected_manifest_sha=EXPECTED_MANIFEST_SHA)
print(data)
print('Original data provenance:', identity['selection']['identity'])


In [ ]:
def launch(*, preflight):
    from uuid import uuid4
    destination = (Path('/kaggle/working') /
        f'preflight_{ARCHITECTURE}_{uuid4().hex[:10]}') if preflight else OUTPUT
    command = [sys.executable, '-m', 'torch.distributed.run', '--master-addr=127.0.0.1', '--master-port=29501',
        '--nproc_per_node', str(NUM_GPUS), str(BUNDLE_ROOT / 'train_kaggle.py'),
        '--bundle-root', str(BUNDLE_ROOT), '--architecture', ARCHITECTURE,
        '--output-dir', str(destination), '--micro-batch', str(MICRO_BATCH),
        '--global-batch', str(GLOBAL_BATCH), '--learning-rate', str(LEARNING_RATE),
        '--precision', PRECISION, '--expected-manifest-sha', EXPECTED_MANIFEST_SHA]
    if preflight:
        command.append('--preflight')
    elif RESUME_FROM:
        command += ['--resume-from', str(RESUME_FROM)]
    if USE_COMPILE:
        command.append('--torch-compile')
    if not GRADIENT_CHECKPOINTING:
        command.append('--no-gradient-checkpointing')
    environment = os.environ | {'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2',
        'TOKENIZERS_PARALLELISM': 'false', 'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'Agg',
        'ACCELERATE_DYNAMO_USE_DYNAMIC': 'true', 'HF_HUB_OFFLINE': '1', 'HF_DATASETS_OFFLINE': '1'}
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, env=environment, check=True)

# Реальные полные модели на GPU,2 optimizer steps, finite loss/gradients,
# validation, checkpoint и экспорт. Ошибка остановит notebook до полного обучения.
launch(preflight=True)


In [ ]:
# Полный эксперимент: одна эпоха на точной фиксированной выборке.
launch(preflight=False)


In [ ]:
from IPython.display import Image, display
result = json.loads((OUTPUT / 'complete.json').read_text())
display(result)
display(Image(filename=str(OUTPUT / 'training.png')))
print('Финальная модель:', OUTPUT / 'final')
print('Checkpoints:', sorted(p.name for p in OUTPUT.glob('checkpoint-*')))
print('Метрики:', OUTPUT / 'complete.json')


Полный запуск сохраняет HF-веса и tokenizer в `final/`, полные
`checkpoint-N` с optimizer/scheduler/RNG и `run_manifest.json`,
`log_history.json`, `training.png`, `complete.json`.
Для автономного запуска выберите **Save Version → Save & Run All**
вместо ручного Run All: Kaggle выполнит notebook и сохранит output.
При разрыве сессии прикрепите сохранённый output как input
и укажите `RESUME_FROM`; данные и настройки должны совпадать.

Copy-веса загружаются после `import models` через
`AutoModelForSeq2SeqLM.from_pretrained(...)` (тип `copy_bart`).
Для SDPA после загрузки T5 вызовите `enable_t5_sdpa(model)` из
`models.t5_attention`. Генерация из обоих моделей поддерживает beam.
Validation loss не является pass@1; после отбора отдельно проверяем
качество кода и лимит180 секунд на T4 с beam4.
